# 1. EDA

### 현장 설명

수도권 소재 **B2C 이커머스 풀필먼트센터**입니다.

- 연면적 약 15,000평 규모
- 하루 평균 수천 건 규모의 주문을 처리
- 6개 Zone(ZA~ZF), SKU 500종, 작업자 약 45명, 지게차 6대 운영
- 주요 영업 시간: 매일 **08:00 ~ 20:00** (마감 시간 전후로 일부 잔여 작업 발생)

### 무슨 일이 일어나고 있나요?

최근 **6개월간 정시 출고율(SLA)이 불안정하게 변동**하며, 특정 시기에 크게 악화되는 현상이 감지되었습니다.

그 중에서도 **화~목요일 오후 14:00~17:00** 시간대에 유독 악화되는 패턴이 반복되고 있습니다. 경영진은 문제가 있다는 건 알고 있지만, **정확히 어떤 문제가 가장 중요한지 파악하지 못하고 있습니다.**

> 여러분은 이 센터의 운영 컨설턴트입니다.
데이터를 분석해 문제를 발견하고, 해결책을 제시해야 합니다.
>

6	inventory_snapshot.csv	시점별 재고 + 공간 정보	약 117만	(snapshot_time, sku_id)

In [2]:
# ============================================================
#  inventory_snapshot.csv — EDA (문제 발견 1-1 / 1-2)
#  B2C 풀필먼트센터 재고·공간 스냅샷 탐색적 분석
# ============================================================
#
#  실행: VS Code에서 이 파일 열고 F5 또는 터미널에서 `python eda_inventory.py`
#  사전 설치: pip install pandas numpy matplotlib seaborn
#
#  이 스크립트가 하는 일
#   - 콘솔에 1-1(파일별 주요 발견)에 넣을 수치/이상값을 출력
#   - ./eda_outputs/ 폴더에 1-2(시각화)용 그래프 PNG 저장
#
#  분석 관점(이 파일의 역할):
#   overflow는 특정 시간대 문제가 아니라 "만성적 공간 초과"일 가능성이 큼.
#   따라서 시간 패턴은 '가정'하지 않고 '검증'만 한다.
# ============================================================

In [ ]:
# ------------------------------------------------------------
#  0. 설정 & 라이브러리
# ------------------------------------------------------------
import os
import platform
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ---- 여기만 본인 경로로 수정 ----
CSV_PATH = "inventory_snapshot.csv"
OUT_DIR = "eda_outputs"
# --------------------------------

os.makedirs(OUT_DIR, exist_ok=True)

# 한글 폰트 (OS별 자동)
if platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
elif platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
else:
    plt.rcParams["font.family"] = "NanumGothic"  # 없으면: sudo apt install fonts-nanum
plt.rcParams["axes.unicode_minus"] = False
sns.set_style("whitegrid")

DOW_KR = ["월", "화", "수", "목", "금", "토", "일"]


def save(fig, name):
    path = os.path.join(OUT_DIR, name)
    fig.savefig(path, dpi=130, bbox_inches="tight")
    plt.close(fig)
    print(f"  [저장] {path}")


# ------------------------------------------------------------
#  1. 데이터 로드 & 기본 구조
# ------------------------------------------------------------
print("\n" + "=" * 60)
print(" 1. 데이터 기본 구조")
print("=" * 60)

df = pd.read_csv(CSV_PATH)
df["t"] = pd.to_datetime(df["snapshot_time"], format="%Y-%m-%d %H:%M")
df["date"] = df["t"].dt.date
df["dow"] = df["t"].dt.dayofweek        # 0=월 ... 6=일
df["hour"] = df["t"].dt.hour

print(f"행 x 열       : {df.shape[0]:,} x {df.shape[1]}")
print(f"기간          : {df['t'].min()}  ~  {df['t'].max()}")
print(f"스냅샷 시점 수 : {df['t'].nunique():,}")
print(f"SKU 종류      : {df['sku_id'].nunique()}   (설계상 500종)")
print(f"Zone          : {sorted(df['zone_id'].unique())}")
print(f"Location 수   : {df['location_id'].nunique()}")


# ------------------------------------------------------------
#  2. 데이터 품질 점검 (결측 / 논리 이상)
# ------------------------------------------------------------
print("\n" + "=" * 60)
print(" 2. 데이터 품질 점검")
print("=" * 60)

print("- 결측치 개수:")
print(df.isnull().sum()[lambda s: s > 0] if df.isnull().sum().sum() else "  없음")

# 논리 검증: available_qty == on_hand - allocated 여야 정상
chk = (df["on_hand_qty"] - df["allocated_qty"] - df["available_qty"]).abs()
print(f"- available_qty 정합성 위반 행: {(chk > 0).sum():,}")

# 스냅샷당 기록 수 (SKU 누락 여부)
per_snap = df.groupby("t").size()
print(f"- 스냅샷당 기록 수: 평균 {per_snap.mean():.1f} / 최소 {per_snap.min()} / 최대 {per_snap.max()}")


# ------------------------------------------------------------
#  3. 핵심 변수 분포 (overflow / utilization)  → 1-2 그래프
# ------------------------------------------------------------
print("\n" + "=" * 60)
print(" 3. 핵심 변수 분포")
print("=" * 60)

of_rate = df["overflow_flag"].mean()
print(f"- 전체 overflow 발생률: {of_rate*100:.1f}%  (전체 {len(df):,}행 중 {df['overflow_flag'].sum():,}행)")
print(f"- location_utilization_pct: 평균 {df['location_utilization_pct'].mean():.1f}% "
    f"/ 최대 {df['location_utilization_pct'].max():.0f}%")
print(f"- 100% 초과(용량 초과) 비율: {(df['location_utilization_pct'] > 100).mean()*100:.1f}%")
print(f"- available_space_liters 음수(공간 초과) 최소값: {df['available_space_liters'].min():,.0f} L")

# 3-1) utilization 분포 히스토그램
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(df["location_utilization_pct"].clip(upper=300), bins=60, ax=ax, color="#3b6fb5")
ax.axvline(100, color="red", ls="--", lw=1.5, label="100% (용량 한계)")
ax.set(title="Location 공간 사용률 분포", xlabel="utilization_pct (%, 300 이상은 300으로 표시)", ylabel="빈도")
ax.legend()
save(fig, "01_utilization_dist.png")

# 3-2) overflow 여부에 따른 사용률 차이 (박스플롯)
fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(data=df, x="overflow_flag", y="location_utilization_pct", ax=ax, showfliers=False)
ax.set(title="overflow 여부별 사용률", xlabel="overflow_flag (0=정상, 1=초과)", ylabel="utilization_pct")
save(fig, "02_overflow_vs_util_box.png")


# ------------------------------------------------------------
#  4. 시간 패턴 검증: 요일 x 시간  (SLA 문제와의 연결 확인)
# ------------------------------------------------------------
print("\n" + "=" * 60)
print(" 4. 시간 패턴 (요일 x 시간)")
print("=" * 60)
print(" ※ '화~목 14~17시'에 공간 문제가 몰리는지 '검증'하는 파트")

piv_of = df.pivot_table(index="dow", columns="hour", values="overflow_flag", aggfunc="mean") * 100
piv_ut = df.pivot_table(index="dow", columns="hour", values="location_utilization_pct", aggfunc="mean")
piv_of.index = [DOW_KR[i] for i in piv_of.index]
piv_ut.index = [DOW_KR[i] for i in piv_ut.index]

print("\n[요일 x 시간 overflow 발생률(%)]")
print(piv_of.round(1))

# 4-1) overflow 발생률 히트맵
fig, ax = plt.subplots(figsize=(11, 4.5))
sns.heatmap(piv_of, annot=True, fmt=".1f", cmap="Reds", ax=ax, cbar_kws={"label": "overflow %"})
ax.set(title="요일 x 시간대 overflow 발생률 (%)", xlabel="시각", ylabel="요일")
save(fig, "03_heatmap_overflow.png")

# 4-2) 평균 사용률 히트맵
fig, ax = plt.subplots(figsize=(11, 4.5))
sns.heatmap(piv_ut, annot=True, fmt=".1f", cmap="YlOrRd", ax=ax, cbar_kws={"label": "util %"})
ax.set(title="요일 x 시간대 평균 사용률 (%)", xlabel="시각", ylabel="요일")
save(fig, "04_heatmap_utilization.png")


# ------------------------------------------------------------
#  5. 공간 관점: Zone / Location 별 overflow 집중도
# ------------------------------------------------------------
print("\n" + "=" * 60)
print(" 5. Zone / Location 별 overflow 집중도")
print("=" * 60)

zone_of = df.groupby("zone_id")["overflow_flag"].mean().mul(100).sort_values(ascending=False)
print("\n[Zone별 overflow 발생률(%)]")
print(zone_of.round(1))

loc_of = df.groupby("location_id")["overflow_flag"].mean().mul(100).sort_values(ascending=False)
print("\n[overflow 상위 10개 Location(%)]")
print(loc_of.head(10).round(1))

# 5-1) Zone별 막대
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=zone_of.index, y=zone_of.values, ax=ax, color="#c0504d")
ax.set(title="Zone별 overflow 발생률", xlabel="Zone", ylabel="overflow %")
save(fig, "05_overflow_by_zone.png")

# 5-2) 상위 15개 Location 막대
top_loc = loc_of.head(15)
fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(y=top_loc.index, x=top_loc.values, ax=ax, color="#8064a2")
ax.set(title="overflow 발생률 상위 15개 Location", xlabel="overflow %", ylabel="location_id")
save(fig, "06_overflow_top_locations.png")


# ------------------------------------------------------------
#  6. 시계열 흐름: 일자별 overflow 추이
# ------------------------------------------------------------
print("\n" + "=" * 60)
print(" 6. 일자별 overflow 추이")
print("=" * 60)

daily = df.groupby("date")["overflow_flag"].mean().mul(100)
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(daily.index, daily.values, lw=1, color="#3b6fb5")
ax.set(title="일자별 overflow 발생률 추이 (6개월)", xlabel="날짜", ylabel="overflow %")
save(fig, "07_daily_overflow_trend.png")

print(f"\n- 일자별 overflow 발생률: 평균 {daily.mean():.1f}% / 최소 {daily.min():.1f}% / 최대 {daily.max():.1f}%")

print("\n" + "=" * 60)
print(f" 완료. 그래프는 '{OUT_DIR}/' 폴더에 저장되었습니다.")
print("=" * 60)